# From Cart to Repeat Purchase

This notebook reviews the implemented cosmetics-store analysis. The question is which recorded purchase-journey gaps deserve investigation and what to test next. It reads validated marts and statistical reports; it does not rebuild events or implement a second set of business formulas.

The source covers October 2019-February 2020, UTC. A proposed cart-information experiment has **not** been run.

In [1]:
from pathlib import Path
from html import escape
import os
import sys

import duckdb
from IPython.display import HTML, display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'config/analysis.toml').is_file())
sys.path.insert(0, str(ROOT / 'src'))
from export_to_postgres import load_reports

database = Path(os.environ.get('SHOP_DATABASE', ROOT / 'data/processed/shop.duckdb'))
reports_dir = Path(os.environ.get('SHOP_REPORTS_DIR', ROOT / 'data/reports'))
connection = duckdb.connect(str(database), read_only=True, config={'memory_limit': '1GB', 'threads': '2'})
reports, fingerprints = load_reports(connection, reports_dir)

def table(columns, rows):
    def value(item):
        if item is None:
            return 'NA'
        if isinstance(item, float):
            return f'{item:,.6f}'
        return str(item)
    header = ''.join(f'<th>{escape(column)}</th>' for column in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(value(item))}</td>' for item in row) + '</tr>'
                   for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))

def query(sql):
    result = connection.execute(sql)
    table([column[0] for column in result.description], result.fetchall())

print(f'Validated {len(reports)} reports against {len(fingerprints)} recorded table fingerprints.')

Validated 4 reports against 25 recorded table fingerprints.


## 1. Inputs and measurement choices

All 20,692,840 source rows remain in staging with provenance and flags. Exact purchase deduplication changes positive event-price value by only 0.0561%; quantity is unknown, so duplicates are retained. Events without a session token remain available for user history.

Original session keys can span long periods. The primary episode starts again after an inactivity gap strictly greater than 30 minutes, within `(user_id, user_session)` across all files. An episode is an operational visit definition, not a verified order.

In [2]:
query('SELECT source_file, row_count, size_bytes FROM source_files ORDER BY source_file')

source_file,row_count,size_bytes
2019-Dec.csv,3533286,415302972
2019-Nov.csv,4635837,545839412
2019-Oct.csv,4102283,482542278
2020-Feb.csv,4156682,488799986
2020-Jan.csv,4264752,501792804


## 2. Ordered funnel and any-cart conversion

The full chain requires a view, a cart at/after that view, and a purchase at/after the qualifying cart. It can involve different products. Any-cart conversion also includes cart episodes without a prior view, so the denominators differ. Strict ordering and original source keys are sensitivity versions.

In [3]:
query('''SELECT session_definition, timestamp_rule, view_units, view_cart_units,
    full_chain_units, cart_units, cart_purchase_units, cart_to_purchase_rate,
    purchase_units, purchase_units_without_ordered_chain
    FROM funnel_summary ORDER BY session_definition, timestamp_rule''')

session_definition,timestamp_rule,view_units,view_cart_units,full_chain_units,cart_units,cart_purchase_units,cart_to_purchase_rate,purchase_units,purchase_units_without_ordered_chain
episode_30m,allow_equal,4501112,745696,95850,1049303,122189,0.116448,157595,61745
episode_30m,strict,4501112,744550,95808,1049303,122156,0.116416,157595,61787
source_key,allow_equal,4281001,724984,100939,985813,125459,0.127265,155617,54678
source_key,strict,4281001,724307,100909,985813,125452,0.127257,155617,54708


The primary any-cart rate is **11.64%** (122,189 / 1,049,303). The full chain covers 95,850 of 157,595 purchase-containing episodes. Missing recorded steps do not identify the reason for leaving or prove tracking faults. Episode abandonment does not imply a customer never buys later.

The separate first-view user funnel uses an inclusive seven-day window across episodes. Its primary cohort enters from November and includes only complete windows; it is not the post-cart experiment baseline.

In [4]:
query('''SELECT population, timestamp_rule, view_users, cart_users, full_chain_users,
    view_to_purchase_rate FROM user_funnel_summary ORDER BY population, timestamp_rule''')

population,timestamp_rule,view_users,cart_users,full_chain_users,view_to_purchase_rate
all_mature_first_views,allow_equal,1529295,283387,68921,0.045067
all_mature_first_views,strict,1529295,283207,68913,0.045062
post_warmup,allow_equal,1140964,169884,49385,0.043284
post_warmup,strict,1140964,169731,49382,0.043281


## 3. Cart segments and uncertainty

The segment population is cart episodes starting November-February. Visitor history uses all available activity, including October and sessionless events. Entry price uses positive prices of unique products at the first cart timestamp. October-only terciles fix thresholds at 3.00 and 6.03 source units; missing positive entry prices form Unknown.

The existing report contains user-clustered estimates. Users can contribute multiple episodes and appear in both groups. Descriptive rate intervals are 95%; the two prespecified contrasts use Bonferroni-adjusted 97.5% intervals.

In [5]:
estimates = reports['user_analysis']['segments']['statistics']['estimates']
table(['Family', 'Segment', 'Cart episodes', 'Users', 'Conversion', '95% lower', '95% upper'],
      [[row[key] for key in ('family', 'segment', 'episodes', 'clusters', 'estimate', 'ci_low', 'ci_high')]
       for row in estimates])
contrasts = reports['user_analysis']['segments']['statistics']['contrasts']
table(['Comparison', 'Difference (proportion)', '97.5% lower', '97.5% upper'],
      [[row['group_b'] + ' minus ' + row['group_a'], row['estimate'], row['ci_low'], row['ci_high']]
       for row in contrasts])

Family,Segment,Cart episodes,Users,Conversion,95% lower,95% upper
price_band,high,259472,149242,0.149538,0.147790,0.151287
price_band,low,291119,128287,0.106551,0.105233,0.107869
price_band,middle,254401,127602,0.114634,0.113197,0.116071
price_band,unknown,2568,1779,0.036994,0.029185,0.044802
visitor_type,first_observed,187607,187570,0.130144,0.128622,0.131667
visitor_type,returning,619953,164034,0.120432,0.119175,0.121689


Comparison,Difference (proportion),97.5% lower,97.5% upper
returning minus first_observed,-0.009713,-0.011945,-0.007480
high minus low,0.042987,0.040701,0.045274


Returning minus first observed is **-0.97 percentage points**; high minus low entry price is **+4.30 pp**. These associations do not show that raising prices improves conversion or that returning is harmful. Clustering handles dependence within user IDs; it does not remove confounding or measurement limitations.

## 4. Repeat purchasing and Black Friday

A repeat requires another purchasing episode beginning strictly after the user's first observed purchase. Multiple item rows in the initial episode are not repeats. The main 30/60-day outcomes use the same complete 60-day population. The sensitivity excluding eight users with overlapping/tied purchasing spans is retained in the reports.

In [6]:
query('''SELECT cohort_label, horizon_days, eligible_users, repeat_users, repeat_rate
    FROM cohort_summary WHERE family='calendar_week' AND population='common_horizon'
    ORDER BY entry_start_utc, horizon_days''')
query('''SELECT cohort_label, month_age, cohort_users, has_complete_month, repeat_rate
    FROM monthly_repurchase ORDER BY entry_start_utc, month_age''')

cohort_label,horizon_days,eligible_users,repeat_users,repeat_rate
Nov 11–17,30,5056,788,0.155854
Nov 11–17,60,5056,1064,0.210443
Nov 18–24,30,7894,1041,0.131872
Nov 18–24,60,7894,1481,0.187611
Nov 25–Dec 1,30,6415,799,0.124552
Nov 25–Dec 1,60,6415,1153,0.179735
Dec 2–8,30,4520,473,0.104646
Dec 2–8,60,4520,754,0.166814


cohort_label,month_age,cohort_users,has_complete_month,repeat_rate
2019-11,1,26761,True,0.097829
2019-11,2,26761,True,0.101491
2019-12,1,19702,True,0.085169
2019-12,2,19702,True,0.064562
2020-01,1,20427,True,0.088608
2020-01,2,20427,False,NA
2020-02,1,17866,False,NA
2020-02,2,17866,False,NA


The BF-week 60-day repeat rate is **17.97%**, below the two preceding November weeks and above the following December week. These are calendar cohorts, not randomized promotion exposures. Customer mix, seasonality, discounts, and costs are not controlled, so no causal promotion effect or ROI is claimed.

Calendar M1/M2 are separate later calendar months, not cumulative 30/60-day windows. NA denotes unavailable follow-up; it must not be replaced by zero.

## 5. Prospective experiment and conditional value

Hypothesis: show complete cost and available fulfillment terms in the cart, without a discount. Verify the current interface and investigate reasons for abandonment before launch.

The experimental baseline enrolls each user once at their first eligible cart in November-February, including users with October history. Its outcome is any purchase within seven days. It differs from both episode conversion and the first-view user funnel.

In [7]:
query('''SELECT timestamp_rule, eligible_users, converted_users, conversion_rate,
    mean_value_per_user, mean_value_per_converted_user FROM cart_experiment_summary
    ORDER BY timestamp_rule''')
query('''SELECT absolute_uplift, is_primary, n_per_group, historical_total_days,
    scenario_population, additional_converted_users, additional_positive_purchase_value
    FROM experiment_scenarios ORDER BY absolute_uplift''')

timestamp_rule,eligible_users,converted_users,conversion_rate,mean_value_per_user,mean_value_per_converted_user
allow_equal,277852,72498,0.260923,10.347538,39.657429
strict,277852,72496,0.260916,10.347331,39.657727


absolute_uplift,is_primary,n_per_group,historical_total_days,scenario_population,additional_converted_users,additional_positive_purchase_value
0.000000,False,NA,NA,10000,0.000000,0.000000
0.005000,False,121833,105,10000,50.000000,"1,982.871431"
0.010000,True,30643,28,10000,100.000000,"3,965.742862"
0.020000,False,7751,21,10000,200.000000,"7,931.485724"


The selected resource-based design is **+1 pp**, 30,643 users per arm, two-sided alpha 0.05 and 80% power. Historical replay gives 21 enrollment days plus seven follow-up days. Future traffic must be validated; this MDE is not an economic profitability threshold.

At an assumed +1 pp and unchanged seven-day positive value per converted user, the conditional value scenario is **3,965.74 source units per 10,000 eligible users**. It is not observed uplift, net revenue, profit, or average order value.

Assign users 50/50 before exposure and analyze all eligible assigned users. Fix enrollment and readout in advance. Assignment/exposure and checkout-error, latency, cancellation/refund guardrails still require production instrumentation.

## Decision and reproducibility

Prioritize investigation of cart completion, then test the cost/fulfillment hypothesis if it fits the actual interface. Preserve population and horizon definitions when discussing results. The data supports a diagnostic recommendation and prospective design, not a claim of realized financial impact.

The [README](../README.md) summarizes findings. The [methodology](../docs/methodology.md) records definitions and limitations. The [Metabase overview](../docs/img/dashboard_overview.png), [segments](../docs/img/dashboard_segments.png), [cohorts](../docs/img/dashboard_cohorts.png), and [experiment planning](../docs/img/dashboard_experiment.png) use the same validated reporting outputs.

In [8]:
connection.close()
print('Read-only review complete; no source or analytical tables were changed.')

Read-only review complete; no source or analytical tables were changed.
